<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<h1 style="color: #2E4A66; border-bottom: 3px solid #3B6EA5; padding-bottom: 8px;">MozzareLLM &mdash; gene cluster analysis</h1>
<p style="color: #5A6B7B;"><em>Pathway identification, evidence-grounded gene categorization, and prioritization of understudied genes &mdash; from any clustered genetic screen.</em></p>

For each cluster of genes, the model:

1. **identifies the biological pathway(s)** that explain why the genes cluster together — or states that no coherent pathway exists (a valid outcome, not a failure);
2. **categorizes every gene relative to that pathway** —

| Category | Meaning |
|---|---|
| `ESTABLISHED` | documented role in this cluster's pathway |
| `NOVEL_ROLE` | documented function elsewhere — membership here is the new evidence |
| `UNCHARACTERIZED` | annotation too sparse to judge a relationship |

   each `NOVEL_ROLE` / `UNCHARACTERIZED` call carries an evidence-ladder subclass and a written rationale;
3. **prioritizes the understudied genes** — the follow-up candidates are the deliverable.

Every call is grounded in a per-cluster *evidence bundle* (functional annotations for each gene, optionally your per-gene phenotypic features), and every run writes a complete audit trail:

| Output file | Contents |
|---|---|
| `<screen>_clusters.csv` | one row per cluster — pathway, confidence, category counts, coverage |
| `<screen>_genes.csv` | one row per gene — category, subclass, rationale, evidence |
| `<screen>_clusters.json` | the parsed structured output |
| `traces/cluster_<id>.json` | full per-call record — raw response, tool calls, tokens, cost |

Read the CSVs; audit the JSONs.


<h2 style="border-left: 5px solid #3B6EA5; padding: 4px 0 4px 12px; color: #2E4A66; margin-bottom: 2px;">1 · Setup</h2>

Requires an `ANTHROPIC_API_KEY` in a `.env` file at the repository root.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [ ]:
import os
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from mozzarellm.clients.llm_api_clients import create_client
from mozzarellm.pipeline.screen_analysis import analyze_screen, prepare_screen_bundles

load_dotenv()
client = create_client(
    model="claude-sonnet-5",
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    max_tokens=32000,  # streamed automatically; headroom for large clusters
)

OUTPUT = Path("output")
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")


Analyzing **your own screen** takes the same two calls you'll see in every example below:

1. `prepare_screen_bundles` — point it at your cluster table (CSV/TSV/XLSX; defaults: `gene_symbol`, `cluster` columns). Bundles are cached — re-running skips the annotation lookups.
2. `analyze_screen` — point it at your screen-context JSON (copy `screen_context_template.json` and fill in your assay, perturbation, readout, and clustering; this grounds the model in what your clusters mean).


<h2 style="border-left: 5px solid #3B6EA5; padding: 4px 0 4px 12px; color: #2E4A66; margin-bottom: 2px;">2 · Example: optical pooled screen, with phenotypic features</h2>

Funk et al. 2022, morphological clusters. **What this example shows:** per-gene phenotypic features (`up_features` / `down_features` — the imaging features driven up or down by each knockout) entering the evidence bundles, plus a per-gene perturbation-strength metric. Pass your feature columns and `strength_column` to `prepare_screen_bundles` — that is the whole integration. Strength can be **any** metric (perturbation AUC, e-distance, normalized fate distance): it is converted to a scale-free rank (`"N/M"`, 1 = strongest vs non-targeting controls) so the model reads it identically across screens. The matching reasoning steps enter the prompt **only when the data is present**: features add a bounded cross-check (is the pathway call consistent with the observed phenotype signature — or does the signature flag a low-quality cluster?), and strength adds a cluster-level informativeness verdict (strong / mixed / weak signal vs controls). Neither ever overturns the pathway call. Describe what your features and strength metric mean in `screen_context.json` under `phenotype_readout` — the model reads your description verbatim.

Feature-mode rationales are verbose: on very large clusters (~40 genes) the response can exceed the output ceiling, and the affected cluster then reports partial coverage honestly — `classification_completeness` < 1, the dropped genes in `missed_genes`, and an entry in `errors`. The per-cluster trace shows the truncated response.


In [ ]:
ops_bundles = prepare_screen_bundles(
    screen_name="funk_2022",
    cluster_table=Path("ops") / "funk_2022.csv",
    output_dir=OUTPUT,
    feature_columns=["up_features", "down_features"],
    strength_column="phenotypic_strength",  # any metric (AUC, e-distance, rank) -> scale-free "N/M" ranks
)
ops = analyze_screen(
    screen_name="funk_2022",
    cluster_to_bundle_map=ops_bundles,
    client=client,
    run_dir=OUTPUT / "funk_2022_analysis" / f"run_{STAMP}_cot_feat",
    screen_context_path=Path("ops") / "screen_context.json",
    mode="cot",
    # No flags needed: the feature and strength reasoning steps enter the prompt
    # automatically because the bundles carry the data (include_features /
    # include_strength default to "auto").
)
print(f"cost: ${ops['total_cost_usd']}  errors: {ops['errors']}")
ops["cluster_df"]

In [ ]:
ops["gene_df"].head(15)  # one row per gene, every category


<h2 style="border-left: 5px solid #3B6EA5; padding: 4px 0 4px 12px; color: #2E4A66; margin-bottom: 2px;">3 · Example: DepMap co-essentiality, baseline analysis</h2>

Wainberg et al. 2021, co-essentiality modules. **What this example shows:** the default path — one chain-of-thought call per cluster over annotation evidence alone. Use this shape for any clustering without per-gene phenotype measurements.


In [ ]:
depmap_bundles = prepare_screen_bundles(
    screen_name="wainberg_2021",
    cluster_table=Path("depmap") / "wainberg_2021.csv",
    output_dir=OUTPUT,
)
depmap = analyze_screen(
    screen_name="wainberg_2021",
    cluster_to_bundle_map=depmap_bundles,
    client=client,
    run_dir=OUTPUT / "wainberg_2021_analysis" / f"run_{STAMP}_cot",
    screen_context_path=Path("depmap") / "screen_context.json",
    mode="cot",
)
depmap["cluster_df"]


<h2 style="border-left: 5px solid #3B6EA5; padding: 4px 0 4px 12px; color: #2E4A66; margin-bottom: 2px;">4 · Example: proteomics co-abundance, with literature validation</h2>

Schaffer et al. 2025, protein co-abundance clusters. **What this example shows:** PubMed literature validation (`mcp=True`) — the model is given search tools and a validation step that checks its `NOVEL_ROLE` and `UNCHARACTERIZED` calls against retrieved literature before finalizing. Slower and costlier per cluster; use it when the flagged genes will drive experiments.


In [ ]:
prot_bundles = prepare_screen_bundles(
    screen_name="schaffer_2025",
    cluster_table=Path("proteomics") / "schaffer_2025.csv",
    output_dir=OUTPUT,
)
prot = analyze_screen(
    screen_name="schaffer_2025",
    cluster_to_bundle_map=prot_bundles,
    client=client,
    run_dir=OUTPUT / "schaffer_2025_analysis" / f"run_{STAMP}_cot_mcp",
    screen_context_path=Path("proteomics") / "screen_context.json",
    mode="cot",
    mcp=True,
)
prot["cluster_df"]


<h2 style="border-left: 5px solid #3B6EA5; padding: 4px 0 4px 12px; color: #2E4A66; margin-bottom: 2px;">5 · Customizing the prompts</h2>

Every piece of wording the model sees is a named text in `mozzarellm/prompts/components.py` — open that file to read the prompts. `mozzarellm/prompts/assembly.py` joins them: the default chain for each mode is `DEFAULT_ORDERS`, and the chain-of-thought steps are numbered `STEP 1 - …` at assembly time (so never start your own text with `STEP N -`). The shipped wording is what our benchmarks selected, but nothing forces you to use it: reword any component per run with `component_overrides={key: text}`, or run a different chain with `component_order=[...]`. Validation and the output tables do not depend on the default chain, only on the output format (`O` / `cO`) being included.

See the full assembled system prompt, then inspect one rendered step:


In [ ]:
from mozzarellm.prompts import COMPONENTS, DEFAULT_ORDERS, make_cluster_analysis_system_prompt, render

print(DEFAULT_ORDERS[("cot", False)])  # the default chain-of-thought chain
print(render()["cGCR"])  # one rendered step (the base GCR rules embedded in the cot step)
print(sorted(COMPONENTS))  # every component key you can override


In [ ]:
my_rules = """GENE CATEGORIZATION:
Categorize each gene as ESTABLISHED, NOVEL_ROLE, or UNCHARACTERIZED relative to the
identified pathway. Treat any gene whose only evidence is from non-human model organisms
as NOVEL_ROLE, and cite the organism in the rationale.
"""

custom = analyze_screen(
    screen_name="wainberg_2021",
    cluster_to_bundle_map=depmap_bundles,
    client=client,
    run_dir=OUTPUT / "wainberg_2021_analysis" / f"run_{STAMP}_cot_custom",
    screen_context_path=Path("depmap") / "screen_context.json",
    mode="cot",
    component_overrides={"cGCR": my_rules},
)
custom["cluster_df"]


A different chain, not just different wording: pass `component_order` with the keys you want, in the order you want them. Here the sub-classification and verification steps are dropped for a shorter, cheaper pass. The output step (`cO`) must stay in the chain — it is what the parser reads.


In [ ]:
short_chain = ["CAT", "SC", "cPH", "cGCR", "cO"]  # mission, context, hypothesis, categorization, output

short = analyze_screen(
    screen_name="wainberg_2021",
    cluster_to_bundle_map=depmap_bundles,
    client=client,
    run_dir=OUTPUT / "wainberg_2021_analysis" / f"run_{STAMP}_cot_short",
    screen_context_path=Path("depmap") / "screen_context.json",
    mode="cot",
    component_order=short_chain,
)
short["cluster_df"]


Component keys: `CAT` (mission), `SC` (screen context, injected per screen), `GCR` / `NPR` / `UPR` (categorization and sub-class rules), `PCC` (pathway confidence), `O` (output format); the chain-of-thought steps `cPH` (hypothesis), `cGCR`, `cPri`, `cPSC` (which embed the rules above — overriding `GCR` also rewords `cGCR`), `cVer` (verification), `cO` (output); `LIT` (literature gap-fill, MCP runs; `LITV` is the category-gated validation variant); `cFC` / `cPC` / `cPS` (phenotype steps, present only when the bundles carry the data). Leave `O` / `cO` alone unless you also change the parsing — it is what keeps results machine-readable.


<h2 style="border-left: 5px solid #3B6EA5; padding: 4px 0 4px 12px; color: #2E4A66; margin-bottom: 2px;">6 · Reading the results</h2>

- **Start with `<screen>_clusters.csv`** — the pathway call and confidence per cluster, and how completely the cluster was classified (`classification_completeness`, `missed_genes`).
- **Then `<screen>_genes.csv`, filtered to `category != "ESTABLISHED"`** — the follow-up candidates, each with its evidence subclass and rationale.
- `dominant_process = "No coherent biological pathway"` with empty gene lists is a deliberate abstention.
- `traces/cluster_<id>.json` holds the complete record of any call you want to audit.
